In [ ]:
from pathlib import Path

# download SynPlanner data
data_folder = Path("synplan_data").resolve()
# download_all_data(save_to=data_folder)

# results folder
results_folder = Path("tutorial_results").resolve()
results_folder.mkdir(exist_ok=True)

# input data
# use default filtered data from tutorial folder or replace with custom data prepared with data curation tutorial
# be sure that you use the same reaction dataset from which the reaction rules were extracted 

reaction_rules_path = '../mapping/uspto_full_rules_light.tsv'
# reaction_rules_path = results_folder.joinpath("uspto_reaction_rules.pickle") # needed for both ranking and filtering policy network training

# filtered_data_path = results_folder.joinpath("uspto_filtered.smi") # needed for ranking policy network training
filtered_data_path = "../mapping/uspto_filtered_light.smi"

# output data
ranking_policy_network_folder = results_folder.joinpath("ranking_policy_network")

# output data
ranking_policy_dataset_path = ranking_policy_network_folder.joinpath("ranking_policy_dataset.pt") # the generated training set for ranking network

In [3]:
from collections import defaultdict
import hashlib
import random
from typing import Dict, List

import torch
from torch.utils.data import Subset
from torch_geometric.data.lightning import LightningDataset

from synplan.utils.config import PolicyNetworkConfig
from synplan.ml.training.supervised import run_policy_training
from synplan.ml.training.preprocessing import RankingPolicyDataset
from synplan.utils.logging import DisableLogger, HiddenPrints

training_config = PolicyNetworkConfig(
    policy_type="ranking",  # the type of policy network
    num_conv_layers=5,  # the number of graph convolutional layers in the network
    vector_dim=512,  # the dimensionality of the final embedding vector
    learning_rate=0.0008,  # the learning rate for the training process
    dropout=0.4,  # the dropout rate
    num_epoch=100,  # the number of epochs for training
    batch_size=100,
)  # the size of training batch of input data


Number of reactions processed: 0 [00:00]

Total examples used: 947632
Split sizes -> train: 877184 (0.926), val: 35224 (0.037), test: 35224 (0.037)
Templates total: 47057 | templates with holdout: 6583


In [4]:
from collections import defaultdict
import random
from torch.utils.data import Subset
from torch_geometric.data.lightning import LightningDataset

def create_ranking_policy_dataset_stratified(
    reaction_rules_path: str,
    reactions_path: str,
    output_path: str,
    batch_size: int = 100,
    train_ratio: float = 0.90,
    val_ratio: float = 0.05,
    test_ratio: float = 0.05,
    min_count_for_holdout: int = 20,
    deduplicate_identical: bool = False,  # fast path
    seed: int = 42,
):
    if abs((train_ratio + val_ratio + test_ratio) - 1.0) > 1e-8:
        raise ValueError("train_ratio + val_ratio + test_ratio must equal 1.0")

    with DisableLogger(), HiddenPrints():
        full_dataset = RankingPolicyDataset(
            reaction_rules_path=reaction_rules_path,
            reactions_path=reactions_path,
            output_path=output_path,
        )

    # Fast label extraction without per-item graph materialization
    y_rules = full_dataset._data.y_rules
    labels = y_rules.view(-1).tolist() if y_rules.dim() > 1 else y_rules.tolist()

    # If dedup is off, no candidate index filtering needed
    if deduplicate_identical:
        candidate_indices = []
        seen = set()
        for idx in range(len(labels)):
            fp = _graph_fingerprint(full_dataset[idx])  # keep your existing dedup logic
            if fp in seen:
                continue
            seen.add(fp)
            candidate_indices.append(idx)
    else:
        candidate_indices = range(len(labels))

    buckets = defaultdict(list)
    for idx in candidate_indices:
        buckets[labels[idx]].append(idx)

    rng = random.Random(seed)
    train_indices, val_indices, test_indices = [], [], []

    for idxs in buckets.values():
        idxs = idxs[:]  # keep original bucket list intact
        rng.shuffle(idxs)
        n_t = len(idxs)

        if n_t < min_count_for_holdout:
            train_indices.extend(idxs)
            continue

        n_val = max(1, int(round(n_t * val_ratio)))
        n_test = max(1, int(round(n_t * test_ratio)))
        n_train = n_t - n_val - n_test

        if n_train < 1:
            train_indices.extend(idxs)
            continue

        train_indices.extend(idxs[:n_train])
        val_indices.extend(idxs[n_train:n_train + n_val])
        test_indices.extend(idxs[n_train + n_val:])

    datamodule = LightningDataset(
        Subset(full_dataset, train_indices),
        Subset(full_dataset, val_indices),
        Subset(full_dataset, test_indices),
        batch_size=batch_size,
        pin_memory=True,
        drop_last=True,
    )

    total = len(train_indices) + len(val_indices) + len(test_indices)
    print(f"Total examples used: {total}")
    print(
        f"Split sizes -> train: {len(train_indices)} ({len(train_indices)/total:.3f}), "
        f"val: {len(val_indices)} ({len(val_indices)/total:.3f}), "
        f"test: {len(test_indices)} ({len(test_indices)/total:.3f})"
    )

    return datamodule, {
        "train_indices": train_indices,
        "val_indices": val_indices,
        "test_indices": test_indices,
    }

datamodule, split_indices = create_ranking_policy_dataset_stratified(
    reaction_rules_path=reaction_rules_path,
    reactions_path=filtered_data_path,
    output_path=ranking_policy_dataset_path,
    batch_size=training_config.batch_size,
    train_ratio=0.90,
    val_ratio=0.05,
    test_ratio=0.05,
    min_count_for_holdout=20,
    deduplicate_identical=False,
    seed=42,
)

Total examples used: 947632
Split sizes -> train: 877184 (0.926), val: 35224 (0.037), test: 35224 (0.037)


In [5]:
run_policy_training(
    datamodule,  # the prepared data module for training
    config=training_config,  # the training configuration
    results_path=ranking_policy_network_folder,
)  # path to save the training results

Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name        | Type           | Params | Mode  | FLOPs
---------------------------------------------------------------
0 | embedder    | GraphEmbedding | 1.3 M  | train | 0    
1 | y_predictor | Linear         | 24.2 M | train | 0    
---------------------------------------------------------------
25.5 M    Trainable params
0         Non-trainable params
25.5 

Weight decoupling enabled in AdaBelief
Rectification enabled in AdaBelief


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]


Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1